## Homework

In this homework, we'll take the ride duration prediction model
that we deployed in batch mode in homework 4 and improve the 
reliability of our code with unit and integration tests. 

You'll find the starter code in the [homework](https://github.com/DataTalksClub/mlops-zoomcamp/tree/main/cohorts/2025/06-best-practices/homework/) directory.

## Q1. Refactoring

Before we can start covering our code with tests, we need to 
refactor it. We'll start by getting rid of all the global variables. 

* Let's create a function `main` with two parameters: `year` and
`month`.
* Move all the code (except `read_data`) inside `main`
* Make `categorical` a parameter for `read_data` and pass it inside `main`

Now we need to create the "main" block from which we'll invoke
the main function. How does the `if` statement that we use for
this looks like? 


Hint: after refactoring, check that the code still works. Just run it e.g. for March 2023 and see if it finishes successfully. 

To make it easier to run it, you can write results to your local
filesystem. E.g. here:

```python
output_file = f'taxi_type=yellow_year={year:04d}_month={month:02d}.parquet'
```

In [1]:
import inspect
import batch
print(inspect.getsource(batch.main))

def main(year, month):
    input_file = get_input_path(year, month)
    output_file = get_output_path(year, month)

    if not output_file.startswith('s3://'):
        os.makedirs(os.path.dirname(output_file) or '.', exist_ok=True)

    with open('model.bin', 'rb') as f_in:
        dv, lr = pickle.load(f_in)

    categorical = ['PULocationID', 'DOLocationID']

    df = read_data(input_file, categorical)
    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

    dicts = df[categorical].to_dict(orient='records')
    X_val = dv.transform(dicts)
    y_pred = lr.predict(X_val)

    print('predicted mean duration:', y_pred.mean())
    print('predicted sum duration:', y_pred.sum())

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['predicted_duration'] = y_pred

    save_data(df_result, output_file)



In [2]:
!sed -n '/^if __name__/,$p' batch.py

if __name__ == '__main__':
    year = int(sys.argv[1])
    month = int(sys.argv[2])
    main(year, month)


In [3]:
# check the refactored script still works on March 2023
%run batch.py 2023 3

predicted mean duration: 14.203865642696089
predicted sum duration: 47103086.50615905


**Answer:** if __name__ == '__main__':

## Q2. Installing pytest

Now we need to install `pytest`:

```bash
pipenv install --dev pytest
```

Next, create a folder `tests` and create two files. One will be
the file with tests. We can name it `test_batch.py`. 

What should be the other file? 

Hint: to be able to test `batch.py`, we need to be able to
import it. Without this other file, we won't be able to do it.

In [4]:
!ls tests

__init__.py   __pycache__   test_batch.py


**Answer:** __init__.py

## Q3. Writing first unit test

Now let's cover our code with unit tests.

We'll start with the pre-processing logic inside `read_data`.

It's difficult to test right now because first reads
the file and then performs some transformations. We need to split this 
code into two parts: reading (I/O) and transformation. 

So let's create a function `prepare_data` that takes in a dataframe 
(and some other parameters too) and applies some transformation to it.

(That's basically the entire `read_data` function after reading 
the parquet file)

Now create a test and use this as input:

```python
data = [
    (None, None, dt(1, 1), dt(1, 10)),
    (1, 1, dt(1, 2), dt(1, 10)),
    (1, None, dt(1, 2, 0), dt(1, 2, 59)),
    (3, 4, dt(1, 2, 0), dt(2, 2, 1)),      
]

columns = ['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime']
df = pd.DataFrame(data, columns=columns)
```

Where `dt` is a helper function:

```python
from datetime import datetime

def dt(hour, minute, second=0):
    return datetime(2023, 1, 1, hour, minute, second)
```

Define the expected output and use the assert to make sure 
that the actual dataframe matches the expected one.

Tip: When you compare two Pandas DataFrames, the result is also a DataFrame.
The same is true for Pandas Series. Also, a DataFrame could be turned into a list of dictionaries.  

How many rows should be there in the expected dataframe?

* 1
* 2
* 3
* 4

In [5]:
print(inspect.getsource(batch.prepare_data))

def prepare_data(df, categorical):
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')

    return df



In [6]:
!cat tests/test_batch.py

from datetime import datetime

import pandas as pd

from batch import prepare_data


def dt(hour, minute, second=0):
    return datetime(2023, 1, 1, hour, minute, second)


def test_prepare_data():
    data = [
        (None, None, dt(1, 1), dt(1, 10)),
        (1, 1, dt(1, 2), dt(1, 10)),
        (1, None, dt(1, 2, 0), dt(1, 2, 59)),
        (3, 4, dt(1, 2, 0), dt(2, 2, 1)),
    ]
    columns = ['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime']
    df = pd.DataFrame(data, columns=columns)

    actual = prepare_data(df, ['PULocationID', 'DOLocationID'])

    expected = [
        {'PULocationID': '-1', 'DOLocationID': '-1',
         'tpep_pickup_datetime': dt(1, 1), 'tpep_dropoff_datetime': dt(1, 10), 'duration': 9.0},
        {'PULocationID': '1', 'DOLocationID': '1',
         'tpep_pickup_datetime': dt(1, 2), 'tpep_dropoff_datetime': dt(1, 10), 'duration': 8.0},
    ]

    assert len(actual) == 2
    assert actual.to_dict(orient='records') == expected


In [7]:
import pytest

pytest.main(['tests', '-v'])

============================= test session starts ==============================
platform darwin -- Python 3.11.17, pytest-9.1.1, pluggy-1.7.0 -- /Users/giahanpham/Documents/trm_learning/zoomcamp/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/giahanpham/Documents/trm_learning/zoomcamp/module 6/homework
plugins: Faker-40.43.0, anyio-4.15.1, platformdirs-4.13.5
collecting ... collected 1 item

tests/test_batch.py::test_prepare_data PASSED                            [100%]

============================== 1 passed in 0.05s ===============================


<ExitCode.OK: 0>

**Answer:** 2

## Q4. Mocking S3 with Localstack 

Now let's prepare for an integration test. In our script, we 
write data to S3. So we'll use Localstack to mimic S3.

First, let's run Localstack with Docker compose. Let's create a 
`docker-compose.yaml` file with just one service: localstack. Inside
localstack, we're only interested in running S3. 

Start the service and test it by creating a bucket where we'll
keep the output. Let's call it "nyc-duration".

With AWS CLI, this is how we create a bucket:

```bash
aws s3 mb s3://nyc-duration
```

Then we need to check that the bucket was successfully created. With AWS, this is how we typically do it:

```bash
aws s3 ls
```

In both cases we should adjust commands for localstack. What option do we need to use for such purposes?

* `--backend-store-uri`
* `--profile`
* `--endpoint-url`
* `--version`

In [8]:
!cat docker-compose.yaml

services:
  localstack:
    image: localstack/localstack:3.8
    ports:
      - "4566:4566"
    environment:
      - SERVICES=s3


In [9]:
!docker compose up -d

[+] up 0/1
 ⠋ Network homework_default Creating                                        0.0s
[+] up 2/2
 ✔ Network homework_default        Created                                  0.0s
 ✔ Container homework-localstack-1 Created                                  0.1s
[+] up 1/2

 ⠙ Container homework-localstack-1 Starting                                 0.2s
[+] up 2/2

 ✔ Container homework-localstack-1 Started                                  0.2s


In [10]:
import os

# Localstack accepts any keys; the checksum settings are needed for newer boto3 versions

os.environ['AWS_ACCESS_KEY_ID'] = 'test'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'test'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
os.environ['AWS_REQUEST_CHECKSUM_CALCULATION'] = 'when_required'
os.environ['AWS_RESPONSE_CHECKSUM_VALIDATION'] = 'when_required'

In [11]:
#using boto3
import boto3

s3 = boto3.client('s3', endpoint_url='http://localhost:4566')
s3.create_bucket(Bucket='nyc-duration')
print('buckets:', [b['Name'] for b in s3.list_buckets()['Buckets']])

buckets: ['nyc-duration']


## Make input and output paths configurable

Right now the input and output paths are hardcoded, but we want
to change it for the tests. 

One of the possible ways would be to specify `INPUT_FILE_PATTERN` and `OUTPUT_FILE_PATTERN` via the env 
variables. Let's do that:


```bash
export INPUT_FILE_PATTERN="s3://nyc-duration/in/{year:04d}-{month:02d}.parquet"
export OUTPUT_FILE_PATTERN="s3://nyc-duration/out/{year:04d}-{month:02d}.parquet"
```

And this is how we can read them:

```python
def get_input_path(year, month):
    default_input_pattern = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet'
    input_pattern = os.getenv('INPUT_FILE_PATTERN', default_input_pattern)
    return input_pattern.format(year=year, month=month)


def get_output_path(year, month):
    default_output_pattern = 's3://nyc-duration-prediction-alexey/taxi_type=fhv/year={year:04d}/month={month:02d}/predictions.parquet'
    output_pattern = os.getenv('OUTPUT_FILE_PATTERN', default_output_pattern)
    return output_pattern.format(year=year, month=month)


def main(year, month):
    input_file = get_input_path(year, month)
    output_file = get_output_path(year, month)
    # rest of the main function ... 
```

In [12]:
print(inspect.getsource(batch.get_input_path))
print(inspect.getsource(batch.get_output_path))

def get_input_path(year, month):
    default_input_pattern = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet'
    input_pattern = os.getenv('INPUT_FILE_PATTERN', default_input_pattern)
    return input_pattern.format(year=year, month=month)

def get_output_path(year, month):
    default_output_pattern = 'output/taxi_type=yellow_year={year:04d}_month={month:02d}.parquet'
    output_pattern = os.getenv('OUTPUT_FILE_PATTERN', default_output_pattern)
    return output_pattern.format(year=year, month=month)



## Reading from Localstack S3 with Pandas

So far we've been reading parquet files from S3 with using
pandas `read_parquet`. But this way we read it from the
actual S3 service. Now we need to replace it with our localstack
one.

For that, we need to specify the endpoint url:

```python
options = {
    'client_kwargs': {
        'endpoint_url': S3_ENDPOINT_URL
    }
}

df = pd.read_parquet('s3://bucket/file.parquet', storage_options=options)
```

Let's modify our `read_data` function:

- check if `S3_ENDPOINT_URL` is set, and if it is, use it for reading
- otherwise use the usual way

In [13]:
print(inspect.getsource(batch.get_storage_options))
print(inspect.getsource(batch.read_data))

def get_storage_options():
    """Point pandas/s3fs at Localstack when S3_ENDPOINT_URL is set."""
    s3_endpoint_url = os.getenv('S3_ENDPOINT_URL')
    if s3_endpoint_url:
        return {'client_kwargs': {'endpoint_url': s3_endpoint_url}}
    return None

def read_data(filename, categorical):
    df = pd.read_parquet(filename, storage_options=get_storage_options())
    return prepare_data(df, categorical)



## Q5. Creating test data

Now let's create `integration_test.py`

We'll use the dataframe we created in Q3 (the dataframe for the unit test)
and save it to S3. You don't need to do anything else: just create a dataframe 
and save it.

We will pretend that this is data for January 2023.

Run the `integration_test.py` script. After that, use AWS CLI to verify that the 
file was created. 

Use this snipped for saving the file:

```python
df_input.to_parquet(
    input_file,
    engine='pyarrow',
    compression=None,
    index=False,
    storage_options=options
)
```

What's the size of the file?

* 3620
* 23620
* 43620
* 63620

Note: it's important to use the code from the snippet for saving
the file. Otherwise the size may be different depending on the OS,
engine and compression. Even if you use this exact snippet, the size
of your dataframe may still be a bit off. Just select the closest option.

In [14]:
from datetime import datetime
import pandas as pd

S3_ENDPOINT_URL = 'http://localhost:4566'
options = {'client_kwargs': {'endpoint_url': S3_ENDPOINT_URL}}


def dt(hour, minute, second=0):
    return datetime(2023, 1, 1, hour, minute, second)


data = [
    (None, None, dt(1, 1), dt(1, 10)),
    (1, 1, dt(1, 2), dt(1, 10)),
    (1, None, dt(1, 2, 0), dt(1, 2, 59)),
    (3, 4, dt(1, 2, 0), dt(2, 2, 1)),
]
columns = ['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime']
df_input = pd.DataFrame(data, columns=columns)

input_file = 's3://nyc-duration/in/2023-01.parquet'
df_input.to_parquet(
    input_file,
    engine='pyarrow',
    compression=None,
    index=False,
    storage_options=options
)

In [15]:
# list the files in the bucket and their size in bytes
for obj in s3.list_objects_v2(Bucket='nyc-duration')['Contents']:
    print(obj['Key'], obj['Size'], 'bytes')

in/2023-01.parquet 3269 bytes


**Answer:** 3620

## Q6. Finish the integration test

We can read from our localstack s3, but we also need to write to it.

Create a function `save_data` which works similarly to `read_data`,
but we use it for saving a dataframe. 

Let's run the `batch.py` script for January 2023 (the fake data
we created in Q5). 

We can do that from our integration test in Python: we can use
`os.system` for doing that (there are other options too). 

Now it saves the result to localstack.

The only thing we need to do now is to read this data and 
verify the result is correct. 

What's the sum of predicted durations for the test dataframe?

* 13.08
* 36.28
* 69.28
* 81.08

In [16]:
print(inspect.getsource(batch.save_data))

def save_data(df, filename):
    df.to_parquet(
        filename,
        engine='pyarrow',
        index=False,
        storage_options=get_storage_options()
    )



In [17]:
!cat integration_test.py

import os
import sys
from datetime import datetime

import pandas as pd

# Localstack doesn't support the checksums that new boto3 versions send by default
os.environ.setdefault('AWS_REQUEST_CHECKSUM_CALCULATION', 'when_required')
os.environ.setdefault('AWS_RESPONSE_CHECKSUM_VALIDATION', 'when_required')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'test')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'test')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

S3_ENDPOINT_URL = os.getenv('S3_ENDPOINT_URL', 'http://localhost:4566')
options = {'client_kwargs': {'endpoint_url': S3_ENDPOINT_URL}}

INPUT_FILE_PATTERN = os.getenv('INPUT_FILE_PATTERN', 's3://nyc-duration/in/{year:04d}-{month:02d}.parquet')
OUTPUT_FILE_PATTERN = os.getenv('OUTPUT_FILE_PATTERN', 's3://nyc-duration/out/{year:04d}-{month:02d}.parquet')


def dt(hour, minute, second=0):
    return datetime(2023, 1, 1, hour, minute, second)


# Q5: create the fake January 2023 input and upload it to Localstack
data = [
    (No

In [18]:
%run integration_test.py

predicted mean duration: 18.138625226015364
predicted sum duration: 36.27725045203073
     ride_id  predicted_duration
0  2023/01_0           23.197149
1  2023/01_1           13.080101
sum of predicted durations: 36.27725045203073


**Answer:** 36.28

## Running the test (ungraded)

The rest is ready, but we need to write a shell script for doing 
that. 

Let's do that!

In [19]:
!cat run.sh

#!/usr/bin/env bash
# Runs the full integration test: starts Localstack, creates the bucket, runs the test, cleans up.
set -e
cd "$(dirname "$0")"

export AWS_ACCESS_KEY_ID=test AWS_SECRET_ACCESS_KEY=test AWS_DEFAULT_REGION=us-east-1
export AWS_REQUEST_CHECKSUM_CALCULATION=when_required AWS_RESPONSE_CHECKSUM_VALIDATION=when_required
export S3_ENDPOINT_URL=http://localhost:4566

docker compose up -d
sleep 5
# Create the bucket. Use the aws CLI if installed, otherwise boto3.
if command -v aws >/dev/null 2>&1; then
    aws --endpoint-url="$S3_ENDPOINT_URL" s3 mb s3://nyc-duration || true
else
    python -c "
import boto3
s3 = boto3.client('s3', endpoint_url='$S3_ENDPOINT_URL')
if 'nyc-duration' not in [b['Name'] for b in s3.list_buckets()['Buckets']]:
    s3.create_bucket(Bucket='nyc-duration')
print('bucket nyc-duration ready')
"
fi

# Don't stop on a test failure, so Localstack still gets shut down below.
set +e
python integration_test.py
ERROR_CODE=$?

docker compose down
exit $ERROR_C

In [20]:
!docker compose down

[+] down 0/1
 ⠋ Container homework-localstack-1 Stopping                                 0.1s

 ⠙ Container homework-localstack-1 Stopping                                 0.2s

 ⠹ Container homework-localstack-1 Stopping                                 0.3s

 ⠸ Container homework-localstack-1 Stopping                                 0.4s

 ⠼ Container homework-localstack-1 Stopping                                 0.5s

 ⠴ Container homework-localstack-1 Stopping                                 0.6s

 ⠦ Container homework-localstack-1 Stopping                                 0.7s

 ⠧ Container homework-localstack-1 Stopping                                 0.8s

 ⠇ Container homework-localstack-1 Stopping                                 0.9s

 ⠏ Container homework-localstack-1 Stopping                                 1.0s

 ⠋ Container homework-localstack-1 Stopping                                 1.1s

 ⠙ Container homework-localstack-1 Stopping                                 1.2s

 ⠹ 

## Submit the results

* Submit your results here: https://courses.datatalks.club/mlops-zoomcamp-2025/homework/hw6
* It's possible that your answers won't match exactly. If it's the case, select the closest one.